# Module 7 — Day 2: Query Planning & Routing

**Goal:** Decompose complex multimodal questions into sub-queries, select the required tools and modalities, route each task, and synthesize evidence.

**Prerequisites:**Upload the same sample invoice PDF, `day4_retrieval_results.json`, and the Day 2 extracted-images ZIP.

## 1. Install dependencies
Install PDF parsing, retrieval, image inspection, and graph orchestration packages.

In [1]:
!pip -q install pymupdf pillow pandas scikit-learn langgraph

## 2. Imports and workspace
Create a clean workspace for this notebook.

In [2]:
import json, re, shutil, zipfile
from pathlib import Path
from typing import TypedDict, Optional
import fitz, pandas as pd
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
WORK = Path("module7_day2")
IMAGE_DIR = WORK / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
RECORDS, ROUTING_TRACE = [], []

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 3. Upload Module 6 artifacts
Upload the sample invoice PDF, Day 4 retrieval JSON (optional), and extracted-images ZIP (optional).

In [3]:
from google.colab import files
uploaded = files.upload()
pdfs = [x for x in uploaded if x.lower().endswith(".pdf")]
jsons = [x for x in uploaded if x.lower().endswith(".json")]
zips = [x for x in uploaded if x.lower().endswith(".zip")]
exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
pdf_path = Path(pdfs[0]) if pdfs else None
json_path = next((Path(x) for x in jsons if "day4_retrieval_results" in x.lower()), None)
for z in zips:
    with zipfile.ZipFile(z) as f: f.extractall(IMAGE_DIR)
for x in uploaded:
    if Path(x).suffix.lower() in exts: shutil.copy2(x, IMAGE_DIR / Path(x).name)
image_paths = sorted(p for p in IMAGE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in exts)
print("PDF:", pdf_path, "| Retrieval JSON:", json_path, "| Images:", [p.name for p in image_paths])

Saving day2_extracted_images.zip to day2_extracted_images.zip
Saving day4_retrieval_results.json to day4_retrieval_results.json
Saving sample_multimodal_invoice.pdf to sample_multimodal_invoice.pdf
PDF: sample_multimodal_invoice.pdf | Retrieval JSON: day4_retrieval_results.json | Images: ['figure-1-1.jpg', 'table-1-1.jpg', 'table-1-2.jpg', 'table-2-3.jpg', 'table-2-4.jpg', 'table-2-5.jpg']


## 4. Load document records
Use Module 6 retrieval JSON when available; otherwise extract text page by page from the PDF.

In [17]:
def load_records():
    rows = []
    if json_path:
        raw = json.loads(json_path.read_text(encoding="utf-8"))
        items = raw if isinstance(raw, list) else raw.get("results", raw.get("retrieved_context", []))
        for i, x in enumerate(items, 1):
            if not isinstance(x, dict): continue
            txt = x.get("text") or x.get("content") or x.get("page_content") or ""
            if txt:
                rows.append({"source_id":x.get("source_id",f"M6-{i}"),"source_doc":x.get("source_doc",x.get("source","Module 6")),
                             "page":x.get("page"),"modality":x.get("modality","text"),"text":str(txt)})
    elif pdf_path:
        with fitz.open(pdf_path) as doc:
            for n, page in enumerate(doc, 1):
                txt = page.get_text().strip()
                if txt: rows.append({"source_id":f"PDF-P{n}","source_doc":pdf_path.name,"page":n,"modality":"text","text":txt})
    RECORDS.clear(); RECORDS.extend(rows)
    return {"records":len(rows),"pages":sorted({r["page"] for r in rows if r["page"] is not None})}
print(load_records())
display(pd.DataFrame([{k:r.get(k) for k in ["source_id","source_doc","page","modality"]} for r in RECORDS]))

{'records': 0, 'pages': []}


""


## 5. Define retrieval and metadata tools
The text tool uses TF-IDF. The image tool locates extracted image files; locating an image is not the same as interpreting its visual content.

In [18]:
def retrieve_text(query, top_k=4, page=None):
    corpus = [r for r in RECORDS if page is None or r["page"] == page]
    if not corpus: return {"ok":False,"results":[],"reason":"No records for requested page."}
    vec = TfidfVectorizer(stop_words="english")
    mat = vec.fit_transform([r["text"] for r in corpus] + [query])
    scores = cosine_similarity(mat[-1], mat[:-1]).ravel()
    ids = scores.argsort()[::-1][:top_k]
    return {"ok":True,"results":[{"source_id":corpus[i]["source_id"],"source_doc":corpus[i]["source_doc"],
        "page":corpus[i]["page"],"modality":corpus[i]["modality"],"score":round(float(scores[i]),4),
        "text":corpus[i]["text"][:1000]} for i in ids]}

def search_metadata(page=None, modality=None):
    found = [ {k:r.get(k) for k in ["source_id","source_doc","page","modality"]} for r in RECORDS
        if (page is None or r["page"]==page) and (modality is None or r["modality"]==modality)]
    return {"ok":True,"matches":found}

def find_images(page=None, keywords=None):
    return {"ok":True,"images":[{"filename":p.name,"path":str(p),"page":None} for p in image_paths]}

TOOLS = {"retrieve_text":retrieve_text,"search_metadata":search_metadata,"find_images":find_images}
print("Tools:", list(TOOLS))

Tools: ['retrieve_text', 'search_metadata', 'find_images']


## 6. Analyze query intent
Identify multi-hop comparison, page references, visual terms, and table/total terms.

In [19]:
def analyze_query(query):
    q = query.lower()
    return {
        "comparison": any(w in q for w in ["compare","difference","versus"," vs "]),
        "pages": [int(x) for x in re.findall(r"page\\s+(\\d+)",q)],
        "visual": any(w in q for w in ["chart","diagram","figure","image","visual","graph"]),
        "table": any(w in q for w in ["table","totals","total","amount","line items"]),
        "query": query
    }
example = "Compare the chart on page 2 with the totals table on page 1."
print(analyze_query(example))

{'comparison': True, 'pages': [], 'visual': True, 'table': True, 'query': 'Compare the chart on page 2 with the totals table on page 1.'}


## 7. Decompose the query into tasks
The planner creates a separate task for each required evidence source, with a tool, modality, page, and focused sub-query.

In [20]:
def plan_query(query):
    s = analyze_query(query); pages = s["pages"]; tasks = []
    if s["comparison"] and s["visual"] and s["table"]:
        tasks = [
            {"task_id":"T1","goal":"Locate the chart/visual","query":"chart graph diagram figure visual","page":pages[0] if pages else None,"modality":"image","tool":"find_images"},
            {"task_id":"T2","goal":"Retrieve totals table evidence","query":"totals table total amount sum line items","page":pages[1] if len(pages)>1 else None,"modality":"text","tool":"retrieve_text"}
        ]
    elif s["visual"]:
        tasks = [
            {"task_id":"T1","goal":"Locate relevant visuals","query":query,"page":pages[0] if pages else None,"modality":"image","tool":"find_images"},
            {"task_id":"T2","goal":"Retrieve related text","query":query,"page":pages[0] if pages else None,"modality":"text","tool":"retrieve_text"}
        ]
    else:
        tasks = [{"task_id":"T1","goal":"Retrieve textual evidence","query":query,"page":pages[0] if pages else None,"modality":"text","tool":"retrieve_text"}]
    return {"original_query":query,"signals":s,"tasks":tasks}
plan = plan_query(example)
print(json.dumps(plan,indent=2))

{
  "original_query": "Compare the chart on page 2 with the totals table on page 1.",
  "signals": {
    "comparison": true,
    "pages": [],
    "visual": true,
    "table": true,
    "query": "Compare the chart on page 2 with the totals table on page 1."
  },
  "tasks": [
    {
      "task_id": "T1",
      "goal": "Locate the chart/visual",
      "query": "chart graph diagram figure visual",
      "page": null,
      "modality": "image",
      "tool": "find_images"
    },
    {
      "task_id": "T2",
      "goal": "Retrieve totals table evidence",
      "query": "totals table total amount sum line items",
      "page": null,
      "modality": "text",
      "tool": "retrieve_text"
    }
  ]
}


## 8. Inspect the plan before execution
Review task routing before any tool runs. This is an important safety and debugging checkpoint.

In [21]:
display(pd.DataFrame([{"task":t["task_id"],"goal":t["goal"],"tool":t["tool"],"modality":t["modality"],"page":t["page"],"query":t["query"]} for t in plan["tasks"]]))

,task,goal,tool,modality,page,query
0,T1,Locate the chart/visual,find_images,image,None,chart graph diagram figure visual
1,T2,Retrieve totals table evidence,retrieve_text,text,None,totals table total amount sum line items


## 9. Execute the routed tasks
Dispatch each task to its selected tool and record every input and output.

In [22]:
def execute_plan(plan):
    outputs=[]
    for task in plan["tasks"]:
        if task["tool"]=="retrieve_text":
            result=retrieve_text(task["query"],top_k=4,page=task["page"])
        elif task["tool"]=="find_images":
            result=find_images(page=task["page"],keywords=task["query"])
        else:
            result=search_metadata(page=task["page"],modality=task["modality"])
        outputs.append({"task":task,"observation":result})
        ROUTING_TRACE.append({"task_id":task["task_id"],"tool":task["tool"],"input":task,"output":result})
    return outputs
routed = execute_plan(plan)
print(json.dumps(routed,indent=2)[:6500])

[
  {
    "task": {
      "task_id": "T1",
      "goal": "Locate the chart/visual",
      "query": "chart graph diagram figure visual",
      "page": null,
      "modality": "image",
      "tool": "find_images"
    },
    "observation": {
      "ok": true,
      "images": [
        {
          "filename": "figure-1-1.jpg",
          "path": "module7_day2/images/figure-1-1.jpg",
          "page": null
        },
        {
          "filename": "table-1-1.jpg",
          "path": "module7_day2/images/table-1-1.jpg",
          "page": null
        },
        {
          "filename": "table-1-2.jpg",
          "path": "module7_day2/images/table-1-2.jpg",
          "page": null
        },
        {
          "filename": "table-2-3.jpg",
          "path": "module7_day2/images/table-2-3.jpg",
          "page": null
        },
        {
          "filename": "table-2-4.jpg",
          "path": "module7_day2/images/table-2-4.jpg",
          "page": null
        },
        {
          "filename": "

## 10. Build a provenance-aware evidence ledger
Keep each result attached to the task that requested it. This prevents mixing chart and table evidence during synthesis.

In [23]:
def make_ledger(outputs):
    ledger=[]
    for item in outputs:
        task, obs = item["task"], item["observation"]
        if task["tool"]=="retrieve_text":
            for hit in obs.get("results",[]):
                ledger.append({"task_id":task["task_id"],"type":"text","source_id":hit["source_id"],
                    "source_doc":hit["source_doc"],"page":hit["page"],"content":hit["text"],"score":hit["score"]})
        elif task["tool"]=="find_images":
            for im in obs.get("images",[]):
                ledger.append({"task_id":task["task_id"],"type":"image_file","source_id":im["filename"],
                    "source_doc":None,"page":im["page"],"content":im["path"],"score":None})
    return ledger
ledger = make_ledger(routed)
display(pd.DataFrame(ledger))

,task_id,type,source_id,source_doc,page,content,score
0,T1,image_file,figure-1-1.jpg,None,None,module7_day2/images/figure-1-1.jpg,None
1,T1,image_file,table-1-1.jpg,None,None,module7_day2/images/table-1-1.jpg,None
2,T1,image_file,table-1-2.jpg,None,None,module7_day2/images/table-1-2.jpg,None
3,T1,image_file,table-2-3.jpg,None,None,module7_day2/images/table-2-3.jpg,None
4,T1,image_file,table-2-4.jpg,None,None,module7_day2/images/table-2-4.jpg,None
5,T1,image_file,table-2-5.jpg,None,None,module7_day2/images/table-2-5.jpg,None


## 11. Synthesize only supported evidence
The answer distinguishes retrieved text from merely located images. It does not invent chart values or claim VLM interpretation.

In [26]:
def synthesize(query, ledger):
    lines=[f"Question: {query}","","Evidence:"]
    texts=[e for e in ledger if e["type"]=="text"]
    imgs=[e for e in ledger if e["type"]=="image_file"]
    for e in texts:
        lines.append(f"- [{e['source_id']}, page {e['page']}] {e['content'][:400]}")
    if not texts: lines.append("- No matching text evidence retrieved.")
    if imgs:
        lines.append("- Located image files: "+", ".join(e["source_id"] for e in imgs))
        lines.append("- These files have not been visually interpreted in this notebook.")
    else: lines.append("- No extracted image files were available.")
    lines += ["","Conclusion: a true chart-versus-table comparison requires visual interpretation of the chart and table evidence. Do not infer visual values from filenames."]
    return "\\n".join(lines)
answer = synthesize(plan["original_query"],ledger)
print(answer)

Question: Compare the chart on page 2 with the totals table on page 1.\n\nEvidence:\n- No matching text evidence retrieved.\n- Located image files: figure-1-1.jpg, table-1-1.jpg, table-1-2.jpg, table-2-3.jpg, table-2-4.jpg, table-2-5.jpg\n- These files have not been visually interpreted in this notebook.\n\nConclusion: a true chart-versus-table comparison requires visual interpretation of the chart and table evidence. Do not infer visual values from filenames.


## 12. Add a retrieval fallback
If page-constrained retrieval returns no useful match, retry across all pages and record that fallback was used.

In [27]:
def retrieve_with_fallback(query,page=None,top_k=4):
    first=retrieve_text(query,top_k=top_k,page=page)
    useful=first.get("ok") and any(x["score"]>0 for x in first.get("results",[]))
    if useful: return {"attempts":1,"fallback_used":False,"result":first}
    second=retrieve_text(query,top_k=top_k,page=None)
    return {"attempts":2,"fallback_used":True,"result":second}
print(json.dumps(retrieve_with_fallback("invoice total amount supplier",page=999),indent=2)[:2500])

{
  "attempts": 2,
  "fallback_used": true,
  "result": {
    "ok": false,
    "results": [],
    "reason": "No records for requested page."
  }
}


## 13. Model the workflow with LangGraph
Create a simple stateful graph: plan → route tools → build evidence ledger → synthesize. Day 3 will add iterative loops and sufficiency checks.

In [28]:
from langgraph.graph import StateGraph, START, END
class AgentState(TypedDict, total=False):
    question: str
    plan: dict
    outputs: list
    ledger: list
    answer: str

def plan_node(state): return {"plan":plan_query(state["question"])}
def route_node(state):
    out=execute_plan(state["plan"])
    return {"outputs":out,"ledger":make_ledger(out)}
def answer_node(state): return {"answer":synthesize(state["question"],state.get("ledger",[]))}

builder=StateGraph(AgentState)
builder.add_node("plan",plan_node)
builder.add_node("route",route_node)
builder.add_node("synthesize",answer_node)
builder.add_edge(START,"plan")
builder.add_edge("plan","route")
builder.add_edge("route","synthesize")
builder.add_edge("synthesize",END)
agent_graph=builder.compile()
print("Planning and routing graph compiled.")

Planning and routing graph compiled.


## 14. Run the LangGraph workflow
Invoke the graph on the multi-hop comparison question and inspect the plan and answer.

In [29]:
graph_result=agent_graph.invoke({"question":"Compare the chart on page 2 with the totals table on page 1."})
print("PLAN:")
print(json.dumps(graph_result["plan"],indent=2))
print("\\nANSWER:")
print(graph_result["answer"])

PLAN:
{
  "original_query": "Compare the chart on page 2 with the totals table on page 1.",
  "signals": {
    "comparison": true,
    "pages": [],
    "visual": true,
    "table": true,
    "query": "Compare the chart on page 2 with the totals table on page 1."
  },
  "tasks": [
    {
      "task_id": "T1",
      "goal": "Locate the chart/visual",
      "query": "chart graph diagram figure visual",
      "page": null,
      "modality": "image",
      "tool": "find_images"
    },
    {
      "task_id": "T2",
      "goal": "Retrieve totals table evidence",
      "query": "totals table total amount sum line items",
      "page": null,
      "modality": "text",
      "tool": "retrieve_text"
    }
  ]
}
\nANSWER:
Question: Compare the chart on page 2 with the totals table on page 1.\n\nEvidence:\n- No matching text evidence retrieved.\n- Located image files: figure-1-1.jpg, table-1-1.jpg, table-1-2.jpg, table-2-3.jpg, table-2-4.jpg, table-2-5.jpg\n- These files have not been visually inter

## 15. Compare routing for different query types
Observe how a direct text question, a page query, and a multi-hop visual comparison produce different plans.

In [30]:
for q in ["Who is the supplier and what is the invoice total?",
          "Find the content on page 1.",
          "Compare the chart on page 2 with the totals table on page 1."]:
    p=plan_query(q)
    print("\\nQUERY:",q)
    display(pd.DataFrame([{"task":t["task_id"],"tool":t["tool"],"modality":t["modality"],"page":t["page"]} for t in p["tasks"]]))

\nQUERY: Who is the supplier and what is the invoice total?


,task,tool,modality,page
0,T1,retrieve_text,text,None


\nQUERY: Find the content on page 1.


,task,tool,modality,page
0,T1,retrieve_text,text,None


\nQUERY: Compare the chart on page 2 with the totals table on page 1.


,task,tool,modality,page
0,T1,find_images,image,None
1,T2,retrieve_text,text,None


## 16. Save plan, evidence, and routing trace
Save the artifacts so the planner's decisions and tool outputs can be reviewed.

In [31]:
save_data={"query":plan["original_query"],"plan":plan,"evidence_ledger":ledger,"routing_trace":ROUTING_TRACE,"answer":answer}
save_path=WORK/"day2_query_plan_and_routing.json"
save_path.write_text(json.dumps(save_data,indent=2,ensure_ascii=False),encoding="utf-8")
print("Saved:",save_path)

Saved: module7_day2/day2_query_plan_and_routing.json
